#### 01 — Validate Dynamic Workflow Outputs

Inspect the curated installment schema, record count, and payment totals.

In [ ]:
# Define defaults for manual notebook execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")

# Read the existing notebook parameters.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

# Inspect the persisted installment schema and record count.
curated_df = spark.table(f"{namespace}.customers_installments_curated")
curated_df.printSchema()
print(f"Curated installment records: {curated_df.count()}")

# Verify payment counts and amounts.
display(spark.sql(f"""
    SELECT payment_status,
           COUNT(*) AS installment_count,
           SUM(amount_due) AS total_amount_due
    FROM {namespace}.customers_installments_curated
    GROUP BY payment_status
    ORDER BY payment_status
"""))

root
 |-- installment_id: long (nullable = true)
 |-- application_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- due_date: date (nullable = true)
 |-- amount_due: decimal(12,2) (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- due_year: integer (nullable = true)
 |-- due_month: string (nullable = true)
 |-- is_paid: boolean (nullable = true)

Curated installment records: 6


payment_status,installment_count,total_amount_due
PAID,2,5750.00
PENDING,4,12350.00


Confirm that each state report contains exactly the matching
applications from the joined dataset.

In [ ]:
from pyspark.sql import functions as F

applications_df = spark.table(
    f"{namespace}.customers_applications_silver"
)

validation_rows = []

for state in ["PE", "SP", "RJ", "MG", "BA"]:
    expected_df = applications_df.filter(F.col("state") == F.lit(state))

    # Align the original columns before comparing the records.
    actual_df = spark.table(
        f"{namespace}.customers_applications_{state.lower()}_silver"
    ).select(*applications_df.columns)

    assert actual_df.exceptAll(expected_df).count() == 0, (
        f"Unexpected or extra rows for {state}"
    )
    assert expected_df.exceptAll(actual_df).count() == 0, (
        f"Missing rows for {state}"
    )

    validation_rows.append((state, actual_df.count()))

display(spark.createDataFrame(
    validation_rows, ["state", "application_count"]
))
print("All five state reports match the source applications.")

state,application_count
PE,2
SP,1
RJ,1
MG,1
BA,1


All five state reports match the source applications.
